# PnD DPI OpenScience walkthrough

Register a field, get a **GeoID**, grant access, let **TerraPipe** weather flow in as **BITEs**, run a **disease forecast** off those BITEs, publish the **risk** back as a BITE, and roll it up across fields.

Run `make demo` first (brings the stack up). Every cell degrades gracefully if a service is down, so this notebook also executes headless via `make notebook`.

Data path: `Pancake TAP (TerraPipe) -> BITE store -> agstack-pnd reads weather BITEs by GeoID -> publishes risk BITEs`.

In [ ]:
import demo_client as dc
print('hub    ', dc.HUB_URL)
print('node   ', dc.NODE_URL)
print('pancake', dc.PANCAKE_URL)
print('pnd    ', dc.PND_URL)

# A hub JWT (the caller's DPI identity). In the real demo you get this by logging
# in to the hub; for headless runs it may be empty and calls will show 401s.
import os
HUB_TOKEN = os.environ.get('HUB_TOKEN', '')
DEMO_GEOID = os.environ.get('DEMO_GEOID', '')  # filled in after register-field

## Step 0 - Are the services up?

In [ ]:
for name, res in dc.health().items():
    state = 'UP' if res.get('ok') else ('DOWN' if res.get('down') else f"HTTP {res.get('status')}")
    print(f'{name:8} {state}')

## Step 1 - Register a field -> GeoID

Registering the field's boundary (WKT) with the AR2 node returns a content-derived **GeoID** - the field's stable identity across the whole DPI. Two near-identical GPS captures converge to the *same* GeoID by construction.

In [ ]:
field_wkt = 'POLYGON((-121.75 38.54, -121.74 38.54, -121.74 38.55, -121.75 38.55, -121.75 38.54))'
res = dc.post(f'{dc.NODE_URL}/register-field-boundary',
              json={'wkt': field_wkt}, headers=dc.bearer(HUB_TOKEN))
print(res.get('status'), res.get('json') or res.get('error'))
geo = (res.get('json') or {}).get('geoid') or (res.get('json') or {}).get('Geo Id')
if geo:
    DEMO_GEOID = geo
elif not DEMO_GEOID:
    DEMO_GEOID = 'demo-geoid-0001'  # offline placeholder so the rest of the flow runs
print('GeoID:', DEMO_GEOID)

## Step 2 - Grant access to the field

The owner creates a field list and issues an owner **grant** (an SD-JWT credential) over it. This is the consent that authorizes reading the field's exact geometry and (optionally) its grant-gated weather.

In [ ]:
field_grant = None
fl = dc.post(f'{dc.PANCAKE_URL}/fieldlists',
             json={'name': 'demo', 'geoids': [DEMO_GEOID]}, headers=dc.bearer(HUB_TOKEN))
print('fieldlist:', fl.get('status'), fl.get('json') or fl.get('error'))
list_id = (fl.get('json') or {}).get('list_id')
if list_id:
    grant = dc.post(f'{dc.PANCAKE_URL}/grants/issue',
                    json={'list_id': list_id, 'grantee_account': 'self',
                          'purpose': 'owner', 'validity_days': 30},
                    headers=dc.bearer(HUB_TOKEN))
    print('grant:', grant.get('status'))
    field_grant = (grant.get('json') or {}).get('credential')
print('have field grant:', bool(field_grant))

## Step 3 - Ingest weather as BITEs (TAP)

Pancake's TAP worker pulls TerraPipe weather (or the offline `seed` adapter) and writes hourly weather **BITEs** keyed by GeoID. In the compose stack the worker runs on a schedule; here we trigger a single pass for our GeoID.

> Edit `tap_vendors.yaml` and replace `DEMO_GEOID_REPLACE_ME` with the GeoID above, then the worker ingests for your field.

In [ ]:
import subprocess, shlex
# Run one ingest pass inside the pancake-tap container (no-op if compose isn't running).
cmd = 'docker compose exec -T pancake-tap python -m pancake_services.tap.worker --config /app/dpi-demo/tap_vendors.yaml --once'
try:
    out = subprocess.run(shlex.split(cmd), capture_output=True, text=True, timeout=120)
    print(out.stdout[-2000:] or out.stderr[-2000:])
except Exception as e:
    print('skip live ingest:', e)

## Step 4 - Read the weather BITEs back

In [ ]:
headers = dc.bearer(HUB_TOKEN)
if field_grant:
    headers['X-Field-Grant'] = field_grant
wx = dc.get(f'{dc.PANCAKE_URL}/bites',
            params={'geoid': DEMO_GEOID, 'type': 'weather_historical', 'limit': 1},
            headers=headers)
print('status', wx.get('status'), 'count', (wx.get('json') or {}).get('count'))
bites = (wx.get('json') or {}).get('bites', [])
if bites:
    body = bites[0]['Body']['sirup_data']
    print('hours:', len(body.get('timestamps', [])), 'resolution:', body.get('resolution'))

## Step 5 - Run a disease forecast off the BITEs

agstack-pnd resolves the GeoID, reads the weather BITEs (`weather_provider=dpi`), runs the model, and publishes the result back as a `pest_disease` BITE (`publish=true`).

In [ ]:
req = {
    'model_uuid': 'a1b2c3d4-0001-4000-8000-000000000001',  # growing_degree_days (built-in)
    'geo_id': DEMO_GEOID,
    'start_date': '2026-07-01', 'end_date': '2026-07-10',
    'weather_provider': 'dpi', 'publish': True,
}
if field_grant:
    req['field_grant'] = field_grant
fc = dc.post(f'{dc.PND_URL}/api/v1/calculate', json=req)
print('status', fc.get('status'))
j = fc.get('json') or {}
if 'summary' in j:
    print('summary:', j['summary'])
    print('published risk BITE:', j.get('metadata', {}).get('risk_bite_published'))
else:
    print(j or fc.get('error'))

## Step 6 - Read the published risk BITE

In [ ]:
risk = dc.get(f'{dc.PANCAKE_URL}/bites',
              params={'geoid': DEMO_GEOID, 'type': 'pest_disease', 'limit': 1},
              headers=dc.bearer(HUB_TOKEN))
print('status', risk.get('status'), 'count', (risk.get('json') or {}).get('count'))
rb = (risk.get('json') or {}).get('bites', [])
if rb:
    print('model:', rb[0]['Body']['sirup_data'].get('model_name'))

## Step 7 - Aggregate across distinct fields

Population statistics (`% of fields at high risk`) are only correct because AR2 de-duplicates fields: each GeoID counts once.

In [ ]:
agg = dc.post(f'{dc.PND_URL}/api/v1/aggregate', json={'geoids': [DEMO_GEOID]})
print('status', agg.get('status'))
print(agg.get('json') or agg.get('error'))

## Step 8 - Same thing, for an AI agent

Everything above is available to an agent over MCP at `PND_URL/mcp`: the `run_forecast` tool takes the same arguments (`geo_id`, dates, `weather_provider`, `field_grant`, `publish`) and `aggregate_risk_tool` rolls results up. Point any MCP client at that endpoint.

### What just happened

- A field became a **GeoID** (stable, content-derived identity).
- Consent was expressed as a **grant** and enforced on the data plane.
- Weather arrived as **BITEs** via TAP - the model never called a weather API.
- A scientist's model ran and published **risk BITEs** joinable on the GeoID.
- Results aggregated correctly because fields are **de-duplicated**.

See `WHATS_MISSING.md` for the honest gap list before calling this production-ready.